-- Job ID: rmisamplesql_data_YYYYMMDDHHMMSS
```
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
```


# RMI Sample Queries: Data Engineer (GA)

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/googlemaps-samples/insights-samples/blob/main/roads_management_insights/rmi_sample_queries/notebooks/Data_Engineer_Samples.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2Fgooglemaps-samples%2Finsights-samples%2Fmain%2Froads_management_insights%2Frmi_sample_queries%2Fnotebooks%2FData_Engineer_Samples.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/bigquery/import?url=https://github.com/googlemaps-samples/insights-samples/blob/main/roads_management_insights/rmi_sample_queries/notebooks/Data_Engineer_Samples.ipynb">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/bigquery/v1/32px.svg" alt="BigQuery Studio logo"><br> Open in BigQuery Studio
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/googlemaps-samples/insights-samples/blob/main/roads_management_insights/rmi_sample_queries/notebooks/Data_Engineer_Samples.ipynb">
      <img width="32px" src="https://upload.wikimedia.org/wikipedia/commons/9/91/Octicons-mark-github.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>



This notebook contains sample queries for the **Data Engineer** persona, specifically for the **GA** stage.

## 1. Setup

In [ ]:
from google.colab import auth
import pandas as pd

auth.authenticate_user()

project_id = 'your-project-id' #@param {type:"string"}

### Writable Dataset

Several queries in this notebook (e.g., those creating Materialized Views, Models, or Views) require a **writable dataset** within your own project. 
**Note**: The source `LINKED_DATASET_NAME` dataset is a read-only subscription and cannot be used to store new resources.

Run the cell below to create a new dataset (e.g., `rmi_analysis`) in your project if you haven't already.

**Important**: When running queries that create new BigQuery resources (e.g., tables, views, models) outside of these `%%bigquery` magic cells, remember to manually prepend the job ID with `msqlfactory--` for proper tracking. For example: `bq query --job_id=msqlfactory--your-descriptive-job-name ...`

In [ ]:
dataset_id = "rmi_analysis" #@param {type:"string"}
! bq --location=US mk --dataset {project_id}:{dataset_id}

## GA (General Availability)

### de1_materialized_view.sql
**Business Question**: Generate a query to create a clustered materialized table of historical_travel_time for a specific monitored corridor including modern closure and segment columns.


In [ ]:
%%bigquery --project {project_id} df_de1_materialized_view
-- ARCHITECTURAL NOTE: Analytics Hub shared datasets are read-only linked datasets.
-- BigQuery explicitly forbids 'CREATE MATERIALIZED VIEW' directly against linked tables.
-- Instead, we use 'CREATE OR REPLACE TABLE ... CLUSTER BY selected_route_id' with column-level
-- OPTIONS descriptions to materialize query outputs in a writable dataset, providing fast
-- downstream lookups without hitting linked table materialized view restrictions.
-- Replace `your-project.your-dataset` with your target location.

CREATE OR REPLACE TABLE `your-project.your-dataset.massachusetts_avenue_corridor`
(
  selected_route_id STRING OPTIONS(description="Unique identifier for the SelectedRoute resource. Primary correlation key across RMI telemetry datasets."),
  display_name STRING OPTIONS(description="User-provided descriptive name for the route. Intended for human readability in reports and UI dashboards."),
  record_time TIMESTAMP OPTIONS(description="The UTC timestamp representing when the route data was computed. Daily partitioning column for pruning scans."),
  duration_in_seconds FLOAT64 OPTIONS(description="The traffic-aware duration of the route in seconds under observed real-time traffic conditions. NULL during road closures."),
  static_duration_in_seconds FLOAT64 OPTIONS(description="The traffic-unaware (static) duration of the route in seconds under ideal free-flow conditions."),
  route_geometry GEOGRAPHY OPTIONS(description="The traffic-aware optimal polyline geometry of the route as a GEOGRAPHY object (WKT, EPSG:4326)."),
  road_segment_ids ARRAY<STRING> OPTIONS(description="Repeated array of Google Place IDs representing traversed road segments at record_time."),
  has_road_closure BOOLEAN OPTIONS(description="Indicates whether there is a full road closure on the route (consistent with Google Maps live closure status).")
)
PARTITION BY DATE(record_time)
CLUSTER BY selected_route_id
OPTIONS (
  description="A 7-day materialized extract of RMI historical travel time data for monitored corridors, inheriting canonical column descriptions."
) AS
SELECT
  h.selected_route_id,
  h.display_name,
  h.record_time,
  h.duration_in_seconds,
  h.static_duration_in_seconds,
  h.route_geometry,
  h.road_segment_ids,
  h.has_road_closure
FROM `LINKED_DATASET_NAME.historical_travel_time` AS h
JOIN `LINKED_DATASET_NAME.routes_status` AS s USING (selected_route_id)
WHERE h.record_time >= TIMESTAMP_SUB(TIMESTAMP('2026-07-29'), INTERVAL 7 DAY)
  AND s.status = 'STATUS_RUNNING'
  -- Avoid fragile display_name LIKE filters by matching structured route_attributes or corridor tags
  AND (
    JSON_VALUE(s.route_attributes, '$.corridor') = 'massachusetts_avenue'
    OR JSON_VALUE(s.route_attributes, '$.corridor_id') = 'mass_ave'
    OR s.display_name LIKE '%Massachusetts Avenue%'
  )
  AND (h.static_duration_in_seconds > 0 OR h.has_road_closure IS TRUE);


### de2_data_cleaning.sql
**Business Question**: Write a query that produces a "cleaned" active version of the routes_status table, correctly casting route_length and exposing modern routing geometry and segment columns.


In [ ]:
%%bigquery --project {project_id} df_de2_data_cleaning
/*
  PRE-REQUISITE: This query utilizes the custom routeAttribute 'route_length' 
  (intended physical length in meters), which has been pre-configured for 
  all routes in this sample dataset.
*/

CREATE OR REPLACE VIEW `your-project.your-dataset.routes_status_cleaned`
(
  selected_route_id OPTIONS(description="Unique identifier for the SelectedRoute resource. Primary correlation key across RMI telemetry datasets."),
  display_name OPTIONS(description="User-provided descriptive name for the route. Intended for human readability in reports and UI dashboards."),
  status OPTIONS(description="The current operational lifecycle state of the route (STATUS_RUNNING)."),
  validation_error OPTIONS(description="Detailed error code explaining why a route is in STATUS_INVALID (NULL for valid running routes)."),
  dynamic_routing OPTIONS(description="Indicates whether the route is dynamic (true) or fixed (false)."),
  static_route_geometry OPTIONS(description="Polyline geometry of the static route calculated without live traffic from waypoints."),
  road_segment_ids OPTIONS(description="Sequence of Place IDs representing topological road segments along the static route."),
  route_length_meters OPTIONS(description="The pre-computed intended route length in meters, cast to FLOAT64 from custom route_attributes.route_length.")
)
OPTIONS(
  description="A cleaned view of active SelectedRoutes status, ensuring STATUS_RUNNING and exposing typed geometry and topological metadata."
)
AS
SELECT
  selected_route_id,
  display_name,
  status,
  validation_error,
  dynamic_routing,
  static_route_geometry,
  road_segment_ids,
  SAFE_CAST(COALESCE(JSON_VALUE(route_attributes, '$.route_length_meters'), JSON_VALUE(route_attributes, '$.route_length')) AS FLOAT64) AS route_length_meters
FROM `LINKED_DATASET_NAME.routes_status`
WHERE status = 'STATUS_RUNNING';


### de3_sri_flattening.sql
**Business Question**: Transform the latest 30 minutes of nested SRI offset data into a flattened format with spatial progress metrics and quality filters using native integer arithmetic.


In [ ]:
%%bigquery --project {project_id} df_de3_sri_flattening
/*
  BIGQUERY OPTIMIZATION PATTERN: Native Integer Offsets vs. Windowing
  
  Modern RMI datasets provide speed_reading_offsets (start_offset, end_offset, speed).
  This eliminates complex ST_MAKELINE spatial polyline assembly and window function sums:
  
  1. Static Pruning: Resolves 'latest_timestamp' to prune partitions ahead of the main SELECT.
  2. Native Offsets: Computes interval lengths directly via (end_offset - start_offset).
  3. Progress Ratios: Derives start_position_ratio and end_position_ratio directly from 
     start_offset and end_offset without LAG() or rolling SUM() window calculations.
  4. Multi-Tier Congestion: Accurately classifies SLOW, TRAFFIC_JAM, and TRAFFIC_JAM_HEAVY.
*/

-- Step 1: Define the static anchor date to narrow down partitions
DECLARE anchor_date DATE DEFAULT '2026-07-29';

-- Step 2: Find the exact latest timestamp and define the 30-minute window
DECLARE latest_timestamp TIMESTAMP;
SET latest_timestamp = (
  SELECT MAX(record_time)
  FROM `LINKED_DATASET_NAME.recent_roads_data`
  WHERE record_time >= TIMESTAMP(anchor_date)
);

-- Step 3: Execute flattening logic using native integer offsets
WITH base_intervals AS (
  SELECT
    r.selected_route_id,
    r.record_time,
    segment_offset AS interval_index,
    sro.speed AS interval_speed_state,
    sro.start_offset,
    sro.end_offset,
    -- Physical interval length in meters via integer subtraction
    (sro.end_offset - sro.start_offset) AS interval_length_meters,
    ST_LENGTH(r.route_geometry) AS actual_route_length_meters,
    SAFE_CAST(COALESCE(JSON_VALUE(s.route_attributes, '$.route_length_meters'), JSON_VALUE(s.route_attributes, '$.route_length')) AS FLOAT64) AS intended_route_length_meters
  FROM `LINKED_DATASET_NAME.recent_roads_data` AS r
  JOIN `LINKED_DATASET_NAME.routes_status` AS s USING (selected_route_id)
  LEFT JOIN UNNEST(r.speed_reading_offsets) AS sro WITH OFFSET AS segment_offset
  WHERE r.record_time >= TIMESTAMP(anchor_date)
    AND r.record_time > TIMESTAMP_SUB(latest_timestamp, INTERVAL 30 MINUTE)
    AND s.status = 'STATUS_RUNNING'
    AND (r.has_road_closure IS FALSE OR r.has_road_closure IS NULL)
    AND ST_GEOMETRYTYPE(r.route_geometry) = 'ST_LineString'
),
quality_filtered_intervals AS (
  SELECT
    *,
    -- Deviation between intended and actual geometry length
    SAFE_DIVIDE(ABS(actual_route_length_meters - intended_route_length_meters), intended_route_length_meters) AS length_deviation_ratio
  FROM base_intervals
  -- Filter for high-integrity geometries (< 5% length deviation)
  WHERE SAFE_DIVIDE(ABS(actual_route_length_meters - intended_route_length_meters), intended_route_length_meters) < 0.05
)
SELECT
  selected_route_id,
  record_time,
  interval_index,
  interval_speed_state,
  start_offset,
  end_offset,
  interval_length_meters,
  -- Rounded relative positions (0.000 to 1.000) within the trip computed directly from offsets
  ROUND(SAFE_DIVIDE(start_offset, actual_route_length_meters), 3) AS start_position_ratio,
  ROUND(SAFE_DIVIDE(end_offset, actual_route_length_meters), 3) AS end_position_ratio,
  length_deviation_ratio
FROM quality_filtered_intervals
-- Filter for congested intervals (including TRAFFIC_JAM_HEAVY) and exclude short GPS noise
WHERE interval_speed_state IN ('SLOW', 'TRAFFIC_JAM', 'TRAFFIC_JAM_HEAVY')
  AND interval_length_meters >= 5
ORDER BY selected_route_id, record_time, interval_index;


### de4_attribute_extraction.sql
**Business Question**: Write a query that pivots JSON route_attributes into distinct columns, supporting up to 25 custom metadata pairs.


In [ ]:
%%bigquery --project {project_id} df_de4_attribute_extraction
CREATE OR REPLACE VIEW `your-project.your-dataset.routes_enriched_attributes`
(
  selected_route_id OPTIONS(description="Unique identifier for the SelectedRoute resource. Primary correlation key across RMI telemetry datasets."),
  static_route_geometry OPTIONS(description="Polyline geometry of the static route calculated without live traffic from waypoints."),
  region OPTIONS(description="The geographical business region extracted from custom route_attributes.region."),
  tier OPTIONS(description="The service tier (e.g. priority, standard) extracted from custom route_attributes.tier."),
  priority OPTIONS(description="The operational monitoring priority level extracted from custom route_attributes.priority."),
  route_length_meters OPTIONS(description="The intended physical length of the route in meters, cast to FLOAT64 from custom route_attributes.route_length.")
)
OPTIONS(
  description="A denormalized view of SelectedRoute metadata, promoting custom JSON route_attributes (up to 25 key-value pairs) into typed top-level columns with inherited RMI definitions."
) AS
SELECT
  selected_route_id,
  static_route_geometry,
  JSON_VALUE(route_attributes, '$.region') as region,
  COALESCE(JSON_VALUE(route_attributes, '$.tier'), JSON_VALUE(route_attributes, '$.priority'), 'STANDARD_TIER') as tier,
  JSON_VALUE(route_attributes, '$.priority') as priority,
  -- route_attributes supports up to 25 key-value string pairs. Casting to FLOAT64 for numerical analysis.
  SAFE_CAST(COALESCE(JSON_VALUE(route_attributes, '$.route_length_meters'), JSON_VALUE(route_attributes, '$.route_length')) AS FLOAT64) as route_length_meters
FROM `LINKED_DATASET_NAME.routes_status`
WHERE status = 'STATUS_RUNNING'
  -- Geometry integrity: in routes_status, the static polyline column is static_route_geometry
  AND ST_GEOMETRYTYPE(static_route_geometry) = 'ST_LineString';


### de5_freshness_audit.sql
**Business Question**: Which active routes have stopped receiving updates, indicating potential data gaps?


In [ ]:
%%bigquery --project {project_id} df_de5_freshness_audit
/*
  AUDIT GOAL: Identify routes that are 'STATUS_RUNNING' but have no recent 
  records in historical_travel_time. This helps detect routes with 
  insufficient traffic or pipeline latency issues.
*/

CREATE OR REPLACE VIEW `your-project.your-dataset.route_freshness_audit`
(
  selected_route_id OPTIONS(description="Unique identifier for the SelectedRoute resource. Primary correlation key across RMI telemetry datasets."),
  display_name OPTIONS(description="User-provided descriptive name for the route. Intended for human readability in reports and UI dashboards."),
  last_updated OPTIONS(description="The UTC timestamp of the most recent telemetry record found in historical_travel_time."),
  hours_since_last_update OPTIONS(description="The age of the telemetry data in hours relative to the reference audit timestamp.")
)
OPTIONS(
  description="Operational audit view identifying active routes with missing or stale travel time telemetry data."
) AS
WITH freshness AS (
  SELECT
    selected_route_id,
    MAX(record_time) as last_updated
  FROM `LINKED_DATASET_NAME.historical_travel_time`
  -- Scans the full sample month to find the latest record for every route
  WHERE ST_GEOMETRYTYPE(route_geometry) = 'ST_LineString' AND record_time BETWEEN '2026-07-01' AND '2026-07-30'
  GROUP BY 1
)
SELECT
  s.selected_route_id,
  s.display_name,
  f.last_updated,
  -- Using '2026-07-30' as the reference 'Now' for this static sample dataset
  TIMESTAMP_DIFF(TIMESTAMP('2026-07-30'), f.last_updated, HOUR) AS hours_since_last_update
FROM `LINKED_DATASET_NAME.routes_status` s
LEFT JOIN freshness f USING(selected_route_id)
-- Focus on routes that SHOULD be providing data
WHERE s.status = 'STATUS_RUNNING'
ORDER BY hours_since_last_update DESC;


### de7_routes_status_snapshot.sql


In [ ]:
%%bigquery --project {project_id} df_de7_routes_status_snapshot
-- Q7: Automate Daily Snapshot of Routes Status (Scheduled Query)
-- Business Question: How can I automate the historical tracking of my SelectedRoutes' status changes and routing configurations across all 12 metadata columns?
-- Product Stage: GA
-- Estimated Bytes Processed: < 1 MB
-- Metadata: Inherits column descriptions from routes_status and adds snapshot metadata.

/*
  AUTOMATION EXAMPLE: 
  To schedule this snapshot daily at 2 AM UTC using the bq CLI, run:
  
  bq mk \
    --transfer_config \
    --project_id="your-project-id" \
    --data_source=scheduled_query \
    --display_name="Daily RMI Routes Status Snapshot" \
    --target_dataset="your_dataset" \
    --schedule="every 24 hours" \
    --params='{
      "query":"INSERT INTO `your-project.your-dataset.routes_status_history` SELECT CURRENT_TIMESTAMP() as snapshot_time, * FROM `LINKED_DATASET_NAME.routes_status`"
    }'
*/

-- STEP 1: Initialize the partitioned history table with all 12 inherited metadata columns
CREATE TABLE IF NOT EXISTS `your-project.your-dataset.routes_status_history` (
  snapshot_time TIMESTAMP OPTIONS(description="The UTC timestamp when this daily snapshot was captured."),
  selected_route_id STRING OPTIONS(description="Unique identifier for the SelectedRoute resource. Primary correlation key across RMI telemetry datasets."),
  display_name STRING OPTIONS(description="User-provided descriptive name for the route. Intended for human readability in reports and UI dashboards."),
  status STRING OPTIONS(description="Current operational lifecycle state of the route at snapshot time (e.g., STATUS_RUNNING, STATUS_INVALID)."),
  validation_error STRING OPTIONS(description="Detailed error code explaining why the route is in STATUS_INVALID at snapshot time (NULL for valid running routes)."),
  low_road_usage_start_time TIMESTAMP OPTIONS(description="The UTC timestamp when low road usage was first detected during periodic re-validation."),
  route_attributes STRING OPTIONS(description="JSON-formatted flat string of custom key-value metadata attributes (up to 25 key-value pairs)."),
  dynamic_routing BOOLEAN OPTIONS(description="Indicates whether the route is configured for dynamic traffic rerouting (true) or fixed routing (false)."),
  origin GEOGRAPHY OPTIONS(description="Origin waypoint coordinate of the route as a GEOGRAPHY point."),
  destination GEOGRAPHY OPTIONS(description="Destination waypoint coordinate of the route as a GEOGRAPHY point."),
  intermediates ARRAY<GEOGRAPHY> OPTIONS(description="Repeated array of intermediate waypoint coordinates along the route."),
  static_route_geometry GEOGRAPHY OPTIONS(description="Polyline geometry of the static route calculated without live traffic from waypoints."),
  road_segment_ids ARRAY<STRING> OPTIONS(description="Sequence of Place IDs representing topological road segments along the static route.")
)
PARTITION BY DATE(snapshot_time)
CLUSTER BY selected_route_id
OPTIONS (
  description="Daily historical snapshots of SelectedRoutes status, routing geometry, and metadata across all 12 canonical routes_status columns."
);

-- STEP 2: The Periodic Append Logic (Manually executable version)
-- This statement appends the current state of all routes into the history table.
INSERT INTO `your-project.your-dataset.routes_status_history`
SELECT
  CURRENT_TIMESTAMP() as snapshot_time,
  selected_route_id,
  display_name,
  status,
  validation_error,
  low_road_usage_start_time,
  route_attributes,
  dynamic_routing,
  origin,
  destination,
  intermediates,
  static_route_geometry,
  road_segment_ids
FROM `LINKED_DATASET_NAME.routes_status`;


### de6_hourly_preaggregation.sql
**Business Question**: How do we transform high-frequency 2-minute SRI telemetry arrays (720 intervals/day) into lightweight hourly profiles without losing peak traffic congestion events?


In [ ]:
%%bigquery --project {project_id} df_de6_hourly_preaggregation
WITH hourly_stage AS (
  SELECT 
    r.selected_route_id,
    s.display_name,
    EXTRACT(HOUR FROM r.record_time) AS hour_of_day,
    -- Pre-aggregate interval records into the worst-case traffic state per hour
    -- Evaluating modern 4-tier congestion: TRAFFIC_JAM_HEAVY -> TRAFFIC_JAM -> SLOW -> NORMAL
    CASE 
      WHEN LOGICAL_OR(sro.speed = 'TRAFFIC_JAM_HEAVY') THEN 'TRAFFIC_JAM_HEAVY'
      WHEN LOGICAL_OR(sro.speed = 'TRAFFIC_JAM') THEN 'TRAFFIC_JAM'
      WHEN LOGICAL_OR(sro.speed = 'SLOW') THEN 'SLOW'
      ELSE 'NORMAL'
    END AS worst_hourly_speed,
    -- Handle road closures safely: audit whether any closure was observed in this hour
    LOGICAL_OR(r.has_road_closure IS TRUE) AS had_hourly_closure
  FROM `LINKED_DATASET_NAME.recent_roads_data` AS r
  JOIN `LINKED_DATASET_NAME.routes_status` AS s USING (selected_route_id)
  LEFT JOIN UNNEST(r.speed_reading_offsets) AS sro
  -- PAYLOAD OPTIMIZATION NOTE:
  -- Stage 1 consolidates 2-minute interval records into hourly worst-case states via LOGICAL_OR.
  -- Stage 2 executes ARRAY_AGG, capping array length at exactly <= 24 elements per route/day,
  -- reducing downstream network transport and visualization client memory by over 95%.
  WHERE ST_GEOMETRYTYPE(r.route_geometry) = 'ST_LineString'
    AND s.status = 'STATUS_RUNNING'
    AND (r.has_road_closure IS FALSE OR r.has_road_closure IS NULL)
    AND r.record_time >= '2026-07-15 00:00:00' AND r.record_time < '2026-07-16 00:00:00'
  GROUP BY r.selected_route_id, s.display_name, hour_of_day
)
SELECT 
  selected_route_id,
  display_name,
  -- Stage 2: Aggregate the 24 hourly buckets into the final clean array
  ARRAY_AGG(STRUCT(hour_of_day, worst_hourly_speed) ORDER BY hour_of_day) AS hourly_congestion_profile,
  ARRAY_LENGTH(ARRAY_AGG(STRUCT(hour_of_day, worst_hourly_speed))) AS hourly_array_length
FROM hourly_stage
GROUP BY selected_route_id, display_name
ORDER BY selected_route_id
LIMIT 100;
